## Setup

This notebook covers **production concerns**:

- Testing primitives — `TestModel`, `FunctionModel`, `capture_run_messages`.
- Observability — Logfire instrumentation.
- Offline evaluation — `pydantic_evals`, custom evaluators.
- Lifecycle hooks — PII redaction, audit logging.
- Robustness — `FallbackModel`.
- Human-in-the-loop — deferred tools / approvals.
- Deployment — `to_web()`, `to_cli()`, `to_a2a()`.

The tests are running without touching LLMs so unlike previous notebook they are free. Only two cells that are cost-marked **💰** (eval suite, deferred tools) hit the real Anthropic API.

Setup mirrors `03_workflows.ipynb`; the notebook stands alone.

In [1]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Annotated, Literal

from pydantic import BaseModel, Field
from pydantic_ai.models.anthropic import AnthropicModel
from pydantic_ai.providers.anthropic import AnthropicProvider
from pydantic_settings import BaseSettings, SettingsConfigDict
from rich import print as rprint
from rich.markdown import Markdown

PROJECT_ROOT_PATH = Path.cwd().parent.parent


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=PROJECT_ROOT_PATH / ".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )

    anthropic_api_key: str = Field(...)
    model_id: str = Field(default="claude-haiku-4-5-20251001")


settings = Settings()  # type: ignore[call-arg]

model = AnthropicModel(
    settings.model_id,
    provider=AnthropicProvider(api_key=settings.anthropic_api_key),
)


# Inline Answer schema — same shape as 01_intro / 02_capabilities / 03_workflows.
class Citation(BaseModel):
    doc_id: Annotated[str, Field(min_length=1, max_length=100)]
    quote: Annotated[str, Field(min_length=1, max_length=500)] = "n/a"


class Answer(BaseModel):
    text: Annotated[str, Field(min_length=1, max_length=4000)]
    citations: list[Citation] = Field(default_factory=list)
    confidence: Annotated[float, Field(ge=0.0, le=1.0)] = 0.8
    risk_flag: Literal["none", "escalate", "urgent"] = "none"


SUPPORT_INSTRUCTIONS = (
    "You are a helpful customer support assistant. "
    "Always answer in the structured Answer format. "
    "If you are not sure, set confidence below 0.4 and say so. "
    "If the question is sensitive or out of scope, set risk_flag='escalate'. "
    "If the question concerns an urgent issue, set risk_flag='urgent'."
)


rprint(Markdown("# Production"))
rprint("model_id:", settings.model_id)

Production

model_id: claude-haiku-4-5-20251001

## `TestModel` + `agent.override` — free deterministic tests

`TestModel` skips the network entirely — it produces a synthetic `Answer` from your schema with no LLM call. `agent.override(model=...)` swaps any agent's model for the duration of a `with` block, so tests run the same agent the app does.

Combined, you get fast offline tests for every `pytest` run with zero API spend.

In [2]:
from pydantic_ai import Agent
from pydantic_ai.models.test import TestModel

assistant: Agent[str, Answer] = Agent(
    model,
    output_type=Answer,
    deps_type=str,
    name="assistant",
    instructions=SUPPORT_INSTRUCTIONS,
)

# A fixture you control — what the model "would" have returned for a given prompt.
fixture = {
    "text": "The pro_plan is $29.99/month for up to 10 users.",
    "citations": [
        {"doc_id": "subscriptions", "quote": "pro_plan ($29.99/month, 10 users)"}
    ],
    "confidence": 0.9,
    "risk_flag": "none",
}

with assistant.override(model=TestModel(custom_output_args=fixture)):
    result = await assistant.run("What does the pro_plan cost?", deps="Alice")

assert isinstance(result.output, Answer)
assert result.output.confidence == 0.9, "custom_output_args should round-trip"
rprint("output type :", type(result.output).__name__)
rprint("text        :", result.output.text)
rprint("confidence  :", result.output.confidence)
rprint("citations   :", result.output.citations)

output type : Answer

text        : The pro_plan is $29.99/month for up to 10 users.

confidence  : 0.9

citations   :
[Citation(doc_id='subscriptions', quote='pro_plan ($29.99/month, 10 users)')]

## `FunctionModel` — script the exact response

When you need to assert on a *specific* model output (or a specific tool-call sequence), `FunctionModel` lets you return a hand-crafted `ModelResponse`. The function receives the message history and returns whatever response you want — including tool-call parts that exercise downstream logic in your agent.

In [3]:
from pydantic_ai.messages import ModelMessage, ModelResponse, ToolCallPart
from pydantic_ai.models.function import AgentInfo, FunctionModel

canned_answer = {
    "text": "Standard shipping arrives in 3-5 business days.",
    "citations": [
        {"doc_id": "shipping", "quote": "3-5 business days"}
    ],
    "confidence": 0.85,
    "risk_flag": "none",
}


def respond(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
    # output_type=Answer is implemented as a synthetic output tool. We call it
    # with the canned dict; pydantic-ai parses the result into Answer.
    output_tool = info.output_tools[0]
    return ModelResponse(
        parts=[
            ToolCallPart(
                tool_name=output_tool.name,
                args=canned_answer,
                tool_call_id="test-call-1",
            )
        ]
    )


with assistant.override(model=FunctionModel(respond)):
    result = await assistant.run("How long is standard shipping?", deps="Bob")

rprint("text       :", result.output.text)
rprint("confidence :", result.output.confidence)
rprint("citations  :", [(c.doc_id, c.quote) for c in result.output.citations])

text       : Standard shipping arrives in 3-5 business days.

confidence : 0.85

citations  :
[('shipping', '3-5 business days')]

## `capture_run_messages()` — debug a failing run

When a tool keeps raising `ModelRetry` and the run aborts, you need to know *what the model saw*. `capture_run_messages()` collects every message exchanged during the run — including the failed tool calls and the retry feedback — even if the run errors out.

Reach for it when:

- A `ModelRetry` loop exhausts its budget and raises `UnexpectedModelBehavior`.
- A `UsageLimits` ceiling is hit mid-run.
- You're staring at "the model just won't call this tool right" and need the exact prompt the model saw.

In [4]:
from pydantic_ai import capture_run_messages
from pydantic_ai.exceptions import ModelRetry, UnexpectedModelBehavior, UsageLimitExceeded
from pydantic_ai.usage import UsageLimits

debug_agent = Agent[None, str](TestModel(), output_type=str)


@debug_agent.tool_plain
def always_fails(x: int) -> str:
    raise ModelRetry(f"never works for x={x}")


captured: list[ModelMessage] = []
with capture_run_messages() as messages:
    try:
        await debug_agent.run(
            "trigger the tool",
            usage_limits=UsageLimits(request_limit=2),
        )
    except (UnexpectedModelBehavior, UsageLimitExceeded) as exc:
        rprint(f"run aborted (expected): {type(exc).__name__}: {exc}")
    captured = list(messages)

rprint(f"\ncaptured {len(captured)} messages:")
for i, msg in enumerate(captured):
    parts = [type(p).__name__ for p in msg.parts]
    rprint(f"  [{i}] {type(msg).__name__:>14s}  parts={parts}")

tool_calls = [
    p
    for msg in captured
    if isinstance(msg, ModelResponse)
    for p in msg.parts
    if isinstance(p, ToolCallPart)
]
rprint(f"\ntool calls observed: {[tc.tool_name for tc in tool_calls]}")

run aborted (expected): UnexpectedModelBehavior: Tool 'always_fails' exceeded max retries count of 1

captured 4 messages:

[0]   ModelRequest  parts=['UserPromptPart']

[1]  ModelResponse  parts=['ToolCallPart']

[2]   ModelRequest  parts=['RetryPromptPart']

[3]  ModelResponse  parts=['ToolCallPart']

tool calls observed: ['always_fails', 'always_fails']

## Logfire instrumentation

One call wires up tracing for every model request, tool call, and HTTP round-trip. If `LOGFIRE_TOKEN` is set in your environment, traces ship to your Logfire dashboard; otherwise they print to the local console exporter.

The configuration function below is idempotent — safe to call from multiple cells. In production code, put it once in your app's startup.

In [9]:
import logfire

_logfire_configured = False


def configure_logfire() -> None:
    """Configure Logfire once + instrument pydantic-ai and httpx."""
    global _logfire_configured
    if _logfire_configured:
        return
    token = os.environ.get("LOGFIRE_TOKEN")
    logfire.configure(
        token=token,
        service_name="support-assistant",
        send_to_logfire=bool(token),
        console=False,
    )
    logfire.instrument_pydantic_ai()
    logfire.instrument_httpx(capture_all=True)
    _logfire_configured = True


configure_logfire()

destination = "Logfire dashboard" if os.environ.get("LOGFIRE_TOKEN") else "local console exporter"
rprint(f"Logfire configured. Traces destination: {destination}")
rprint("(set LOGFIRE_TOKEN to ship to https://logfire-us.pydantic.dev/)")

Attempting to instrument while already instrumented


Logfire configured. Traces destination: local console exporter

(set LOGFIRE_TOKEN to ship to https://logfire-us.pydantic.dev/)

## `pydantic_evals` — `Case` and `Dataset`

Two primitives:

- **`Case`** — one input + metadata (expected outputs, flags, anything you'll score against later).
- **`Dataset`** — a collection of `Case`s plus a list of evaluators. `await dataset.evaluate(task)` runs the task on every case, scores it with every evaluator, and returns a report.

The cases are defined inline below. Metadata carries `expected_keywords`, `must_escalate`, and a reference `expected_answer` the LLM judge will use.

In [10]:
from typing import Any

from pydantic_evals import Case, Dataset

EvalMetadata = dict[str, Any]

_CASES: list[Case[str, Answer, EvalMetadata]] = [
    Case(
        name="qa-01-price",
        inputs="What does the pro_plan cost?",
        metadata={
            "expected_keywords": ["29.99", "pro_plan"],
            "must_escalate": False,
            "expected_answer": "The pro_plan is $29.99/month, supporting up to 10 users.",
        },
    ),
    # Case(
    #     name="qa-02-return-window",
    #     inputs="How long is the return window?",
    #     metadata={
    #         "expected_keywords": ["30", "days"],
    #         "must_escalate": False,
    #         "expected_answer": "Returns are accepted within 30 days of delivery.",
    #     },
    # ),
    # Case(
    #     name="qa-03-shipping",
    #     inputs="How long does standard shipping take?",
    #     metadata={
    #         "expected_keywords": ["3", "5", "business"],
    #         "must_escalate": False,
    #         "expected_answer": "Standard shipping takes 3-5 business days.",
    #     },
    # ),
    # Case(
    #     name="qa-04-cancellation",
    #     inputs="Can I cancel my subscription any time?",
    #     metadata={
    #         "expected_keywords": ["cancel", "end", "period"],
    #         "must_escalate": False,
    #         "expected_answer": "Yes — cancellation takes effect at the end of the current billing period.",
    #     },
    # ),
    # Case(
    #     name="qa-05-plans",
    #     inputs="What's the difference between basic_plan and pro_plan?",
    #     metadata={
    #         "expected_keywords": ["basic_plan", "pro_plan", "user"],
    #         "must_escalate": False,
    #         "expected_answer": "basic_plan is $9.99/month for 1 user; pro_plan is $29.99/month for up to 10 users.",
    #     },
    # ),
    # Case(
    #     name="qa-06-double-charge",
    #     inputs="I've been double-charged this month, this is really urgent — fix it now!",
    #     metadata={
    #         "expected_keywords": ["refund", "billing"],
    #         "must_escalate": True,
    #         "expected_answer": "Apologise, escalate to billing for an urgent refund investigation.",
    #     },
    # ),
    # Case(
    #     name="qa-07-account-stolen",
    #     inputs="Someone got into my account and is making fraudulent charges — please help urgently!",
    #     metadata={
    #         "expected_keywords": ["account", "security"],
    #         "must_escalate": True,
    #         "expected_answer": "Acknowledge urgency, escalate to security/fraud team immediately.",
    #     },
    # ),
    # Case(
    #     name="qa-08-support-channels",
    #     inputs="What support channels are available for pro tier?",
    #     metadata={
    #         "expected_keywords": ["priority", "phone"],
    #         "must_escalate": False,
    #         "expected_answer": "Pro and enterprise customers get priority phone support 24/7; all tiers get email.",
    #     },
    # ),
]

rprint(f"defined {len(_CASES)} cases inline")
rprint(f"\nsample case  : {_CASES[0].name}")
rprint(f"  inputs     : {_CASES[0].inputs!r}")
rprint("  metadata   :")
for k, v in _CASES[0].metadata.items():
    rprint(f"      {k:>20s}: {v!r}")

defined 1 cases inline

sample case  : qa-01-price

inputs     : 'What does the pro_plan cost?'

metadata   :

expected_keywords: ['29.99', 'pro_plan']

must_escalate: False

expected_answer: 'The pro_plan is $29.99/month, supporting up to 10 users.'

## Three custom `Evaluator`s

Three kinds of evaluator, in order of cost:

- **`KeywordOverlapEvaluator`** — pure, no LLM: fraction of `expected_keywords` present in the answer text. Free.
- **`RiskFlagEvaluator`** — rule-based: `1.0` iff `must_escalate == (risk_flag != "none")`. Penalises both missed escalations *and* false-positive alarms. Free.
- **`LLMJudgeEvaluator`** — a small judge agent grades the candidate against the reference answer. Returns a Pydantic-validated `JudgeScore` so the score is structurally trusted, not just parsed. Hits the API once per case.

In [11]:
from dataclasses import dataclass

from pydantic_evals.evaluators import Evaluator, EvaluatorContext


@dataclass
class KeywordOverlapEvaluator(Evaluator[str, Answer, EvalMetadata]):
    """Fraction of `expected_keywords` that appear in the answer text."""

    evaluation_name = "keyword_overlap"

    def evaluate(self, ctx: EvaluatorContext[str, Answer, EvalMetadata]) -> float:
        expected: list[str] = list((ctx.metadata or {}).get("expected_keywords") or [])
        if not expected:
            return 1.0
        text = ctx.output.text.lower()
        hits = sum(1 for kw in expected if kw.lower() in text)
        return hits / len(expected)


@dataclass
class RiskFlagEvaluator(Evaluator[str, Answer, EvalMetadata]):
    """Reward correctly flagged escalations and correctly-quiet answers."""

    evaluation_name = "risk_flag"

    def evaluate(self, ctx: EvaluatorContext[str, Answer, EvalMetadata]) -> float:
        must_escalate = bool((ctx.metadata or {}).get("must_escalate", False))
        flagged = ctx.output.risk_flag != "none"
        return 1.0 if must_escalate == flagged else 0.0


class JudgeScore(BaseModel):
    """Schema the judge agent must return — keeps the score structurally validated."""

    score: float = Field(ge=0.0, le=1.0)
    reasoning: str = Field(min_length=1, max_length=500)


_JUDGE_AGENT: Agent[None, JudgeScore] | None = None


def _judge_agent() -> Agent[None, JudgeScore]:
    global _JUDGE_AGENT
    if _JUDGE_AGENT is None:
        _JUDGE_AGENT = Agent[None, JudgeScore](
            model,
            output_type=JudgeScore,
            instructions=(
                "You are an evaluation judge. Given a question, a reference answer, "
                "and a candidate answer, return {score: float, reasoning: str} where "
                "`score` is between 0.0 and 1.0. 1.0 means the candidate captures the "
                "reference answer's key facts; 0.0 means it's wrong or unrelated. "
                "Penalise hallucinated specifics. Be strict on numbers."
            ),
        )
    return _JUDGE_AGENT


@dataclass
class LLMJudgeEvaluator(Evaluator[str, Answer, EvalMetadata]):
    evaluation_name = "llm_judge"

    async def evaluate(self, ctx: EvaluatorContext[str, Answer, EvalMetadata]) -> float:
        question = ctx.inputs
        expected = (ctx.metadata or {}).get("expected_answer", "")
        candidate = ctx.output.text
        prompt = (
            f"Question:\n{question}\n\n"
            f"Reference answer:\n{expected}\n\n"
            f"Candidate answer:\n{candidate}\n\n"
            "Score the candidate's match to the reference."
        )
        result = await _judge_agent().run(prompt)
        return float(result.output.score)


dataset = Dataset[str, Answer, EvalMetadata](
    name="04_production-evals",
    cases=_CASES,
    evaluators=[KeywordOverlapEvaluator(), RiskFlagEvaluator(), LLMJudgeEvaluator()],
)
rprint(f"dataset name : {dataset.name}")
rprint(f"cases        : {len(dataset.cases)}")
rprint(f"evaluators   : {[type(e).__name__ for e in dataset.evaluators]}")

dataset name : 04_production-evals

cases        : 1

evaluators   : ['KeywordOverlapEvaluator', 'RiskFlagEvaluator', 'LLMJudgeEvaluator']

## Run the eval suite 💰

**Cost warning:** this hits the real Anthropic API. With 8 cases × (1 task call + 1 judge call) ≈ **16 model requests**. Skip the cell if you're budget-conscious; the report shape is identical whether you run 8 cases or 800.

The eval target is a tiny agent given the price catalogue in its instructions — no tools, no RAG — so don't expect perfect scores. The point is what the report looks like and how the three evaluators score the same output.

In [12]:
eval_agent: Agent[str, Answer] = Agent(
    model,
    output_type=Answer,
    deps_type=str,
    name="eval-agent",
    instructions=(
        SUPPORT_INSTRUCTIONS
        + "\n\nKnown catalogue (USD/month):\n"
        "- basic_plan $9.99 (1 user)\n"
        "- pro_plan $29.99 (10 users)\n"
        "- enterprise_plan $99.99 (unlimited)\n"
        "Standard shipping: 3-5 business days, free on orders > $50.\n"
        "Returns: 30 days from delivery.\n"
        "Cancellation: takes effect at end of billing period."
    ),
)


async def task(question: str) -> Answer:
    result = await eval_agent.run(question, deps="eval-runner")
    return result.output


report = await dataset.evaluate(task, max_concurrency=4)

averages = report.averages()
rprint("\n=== Aggregate scores ===")
if averages and averages.scores:
    for name, value in sorted(averages.scores.items()):
        rprint(f"  {name:>20s}: {value:.3f}")
rprint(f"\ncases run: {len(report.cases)}  failures: {len(report.failures)}")

/Users/dan/Things/job/datasentics/projects/kbc_workshop/pydantic_ai_workshop/.venv/lib/python3.12/site-packages/ric
h/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/dan/Things/job/datasentics/projects/kbc_workshop/pydantic_ai_workshop/.venv/lib/python3.12/site-packages/ric
h/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/dan/Things/job/datasentics/projects/kbc_workshop/pydantic_ai_workshop/.venv/lib/python3.12/site-packages/ric
h/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

=== Aggregate scores ===

keyword_overlap: 1.000

llm_judge: 1.000

risk_flag: 1.000

cases run: 1  failures: 0

## `Hooks` — 22 lifecycle callbacks

Pydantic AI exposes hook points at every stage of a run. You decorate functions onto a `Hooks` instance, then wire it into the agent via `capabilities=[hooks]`. No middleware, no monkey-patching.

The 22 callbacks group into six families:

| Family | Hooks |
|---|---|
| **Run lifecycle** | `before_run`, `after_run`, `run` (wrap), `run_error` |
| **Node lifecycle** | `before_node_run`, `after_node_run`, `node_run` (wrap), `node_run_error` |
| **Event stream** | `run_event_stream`, `event` |
| **Model request** | `before_model_request`, `after_model_request`, `model_request` (wrap), `model_request_error` |
| **Tool preparation** | `prepare_tools` |
| **Tool validation** | `before_tool_validate`, `after_tool_validate`, `tool_validate` (wrap), `tool_validate_error` |
| **Tool execution** | `before_tool_execute`, `after_tool_execute`, `tool_execute` (wrap), `tool_execute_error` |

Below we wire two callbacks on a fresh `Hooks` instance: PII redaction on `before_model_request` and audit logging on `before_tool_execute`. The next two sections implement them for real.

In [13]:
from pydantic_ai import RunContext
from pydantic_ai.capabilities import Hooks

hooks: Hooks[str] = Hooks[str]()


@hooks.on.before_model_request
def _stub_redact(ctx, request_context):
    # Real implementation in the next code cell — this stub just demonstrates the wiring.
    return request_context


@hooks.on.before_tool_execute
async def _stub_audit(ctx: RunContext[str], *, call, tool_def, args):
    # Real implementation two cells down.
    return args


rprint("hooks wired with two callbacks:")
rprint("  before_model_request -> _stub_redact (PII redaction goes here)")
rprint("  before_tool_execute  -> _stub_audit  (audit logging goes here)")

hooks wired with two callbacks:

before_model_request -> _stub_redact (PII redaction goes here)

before_tool_execute  -> _stub_audit  (audit logging goes here)

## PII-redaction hook

A pure regex sweep — emails, phone numbers, and credit-card-shaped digit sequences get replaced before any message reaches the model provider. Three patterns:

- `_EMAIL_RE` — `name@host.tld`.
- `_PHONE_RE` — international-ish: optional `+`, then 9+ digits with spaces/dashes.
- `_CCARD_RE` — 13–19 digits with optional separators (Luhn-shaped; we don't validate the checksum).

The `redact_pii` function is a pure pass over the input text. The `before_model_request` hook walks every outgoing string field and rewrites it. We test the redactor on a sample string here; the hook integration is the same one-liner.

In [14]:
import re

_EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
_PHONE_RE = re.compile(r"\+?\d[\d\s-]{8,}\d")
_CCARD_RE = re.compile(r"\b(?:\d[ -]*?){13,19}\b")

_REDACTIONS: tuple[tuple[re.Pattern[str], str], ...] = (
    (_EMAIL_RE, "<REDACTED-EMAIL>"),
    (_CCARD_RE, "<REDACTED-CARD>"),
    (_PHONE_RE, "<REDACTED-PHONE>"),
)


def redact_pii(text: str) -> str:
    """Apply all configured redactions to a string. Pure function — easy to test."""
    for pattern, replacement in _REDACTIONS:
        text = pattern.sub(replacement, text)
    return text


def redact_obj(value: Any) -> Any:
    """Recursively redact PII inside JSON-shaped tool args (str / list / dict)."""
    if isinstance(value, str):
        return redact_pii(value)
    if isinstance(value, list):
        return [redact_obj(v) for v in value]
    if isinstance(value, dict):
        return {k: redact_obj(v) for k, v in value.items()}
    return value


raw = (
    "Email me at alice@example.com or call +1-555-0100, "
    "card 4242 4242 4242 4242."
)
redacted = redact_pii(raw)
rprint("raw     :", raw)
rprint("redacted:", redacted)
assert "alice@example.com" not in redacted
assert "+1-555-0100" not in redacted
assert "4242 4242 4242 4242" not in redacted
rprint("\nall three PII patterns scrubbed.")

raw     : Email me at alice@example.com or call +1-555-0100, card 4242 4242 4242 4242.

redacted: Email me at <REDACTED-EMAIL> or call <REDACTED-PHONE>, card <REDACTED-CARD>.

all three PII patterns scrubbed.

## Audit-log hook — SQLite writer

A `before_tool_execute` hook can INSERT `(ts, tool_name, user, args_redacted)` into a SQLite table. Auditors can replay exactly what the bot tried to do — and the args are redacted before they hit disk, same regex pass as outgoing messages.

The cell below builds a fresh SQLite DB in `tempfile.gettempdir()`, wires the hook into a tiny `Hooks` instance, runs an agent that calls a price-lookup tool, then SELECTs from the audit log to show what landed.

In [15]:
import json
import sqlite3
import tempfile
from datetime import datetime, timezone

_AUDIT_DB = Path(tempfile.gettempdir()) / "workshop_audit.db"
_AUDIT_DB.unlink(missing_ok=True)

with sqlite3.connect(_AUDIT_DB) as _conn:
    _conn.execute(
        """CREATE TABLE audit_log (
            ts             TEXT NOT NULL,
            tool_name      TEXT NOT NULL,
            user           TEXT,
            args_redacted  TEXT
        )"""
    )


audit_hooks: Hooks[str] = Hooks[str]()


@audit_hooks.on.before_tool_execute
async def audit_tool_call(ctx: RunContext[str], *, call, tool_def, args):
    redacted = redact_obj(args)
    with sqlite3.connect(_AUDIT_DB) as conn:
        conn.execute(
            "INSERT INTO audit_log (ts, tool_name, user, args_redacted) VALUES (?, ?, ?, ?)",
            (
                datetime.now(timezone.utc).isoformat(timespec="seconds"),
                call.tool_name,
                ctx.deps,
                json.dumps(redacted, default=str),
            ),
        )
    return args


_PRICE_LIST = {"basic_plan": 9.99, "pro_plan": 29.99, "enterprise_plan": 99.99}


def lookup_price(item_name: str) -> dict:
    """Look up a product price by canonical name."""
    key = item_name.strip().lower()
    if key not in _PRICE_LIST:
        return {"error": "unknown item", "known": sorted(_PRICE_LIST)}
    return {"item_name": key, "price_usd": _PRICE_LIST[key]}


audit_agent = Agent[str, str](
    model,
    deps_type=str,
    tools=[lookup_price],
    capabilities=[audit_hooks],
    instructions="Use the lookup_price tool for product-price questions.",
)

await audit_agent.run(
    "What does the pro_plan cost? My email is alice@example.com.",
    deps="Alice",
)

with sqlite3.connect(_AUDIT_DB) as conn:
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        "SELECT ts, tool_name, user, args_redacted FROM audit_log ORDER BY ts DESC"
    ).fetchall()

rprint(f"audit DB: {_AUDIT_DB}")
rprint(f"rows    : {len(rows)}\n")
for r in rows:
    rprint(f"  {r['ts']}  {r['tool_name']:>20s}  user={r['user']!r}")
    rprint(f"    args={r['args_redacted']}")

audit DB: /var/folders/2h/0rn53sk962v5pdjbjs_4mplm0000gn/T/workshop_audit.db

rows    : 1

2026-05-12T18:18:06+00:00          lookup_price  user='Alice'

args={"item_name": "pro_plan"}

## `FallbackModel` — transparent failover

Production agents wrap the primary model in `FallbackModel(primary, fallback)`. If the primary 5xx's or rate-limits, the run *transparently* retries on the fallback — no exception, no surfaced error. Cheap SLA insurance.

A common shape: `primary=Sonnet` (better quality, more expensive) and `fallback=Haiku` (cheap and always-on). If Sonnet capacity is constrained or your account hits a rate limit, Haiku takes over and the user never notices.

NOTE: if user never notices why not use Haiku in the first place? Also shouldn't fallback be to different provider or even local model?

In [16]:
from pydantic_ai.models.fallback import FallbackModel

# Construction only — we don't deliberately break Sonnet just to demo the failover.
production_model = FallbackModel(
    AnthropicModel("claude-sonnet-4-5", provider=AnthropicProvider(api_key=settings.anthropic_api_key)),
    AnthropicModel("claude-haiku-4-5-20251001", provider=AnthropicProvider(api_key=settings.anthropic_api_key)),
)

production_assistant: Agent[str, Answer] = Agent(
    production_model,
    output_type=Answer,
    deps_type=str,
    name="production_assistant",
    instructions=SUPPORT_INSTRUCTIONS,
    capabilities=[hooks],
)

rprint("production_model type :", type(production_model).__name__)
rprint("underlying models     :", [type(m).__name__ for m in production_model.models])
rprint("model names           :", [m.model_name for m in production_model.models])
rprint("agent.model is fallback:", isinstance(production_assistant.model, FallbackModel))

production_model type : FallbackModel

underlying models     :
['AnthropicModel', 'AnthropicModel']

model names           :
['claude-sonnet-4-5', 'claude-haiku-4-5-20251001']

agent.model is fallback: True

## Deferred tools — human-in-the-loop approval 💰

Decorate a tool with `requires_approval=True` and include `DeferredToolRequests` in the agent's `output_type`. When the model tries to call that tool, the run **halts** and returns a `DeferredToolRequests` instead of an answer.

You inspect, approve / deny, and resume with `deferred_tool_results=...`. Use this for any tool whose side effects need a human-in-the-loop sign-off — refunds, transfers, account changes, anything irreversible.

The demo below does the full three-step flow: model proposes a refund, we auto-approve, the run resumes and finalises the `Answer`. Hits the real API twice.

In [17]:
from pydantic_ai import DeferredToolRequests, DeferredToolResults

approval_agent: Agent[None, Answer | DeferredToolRequests] = Agent(
    model,
    output_type=[Answer, DeferredToolRequests],
    instructions=(
        "You are a customer support assistant. When a customer asks for a refund, "
        "call the refund_order tool. After the refund completes, summarise the "
        "outcome in the Answer format with confidence and a citation."
    ),
)


@approval_agent.tool_plain(requires_approval=True)
def refund_order(order_id: str, amount_usd: float, reason: str) -> str:
    """Issue a refund for an order. Requires human approval."""
    return (
        f"Refunded {amount_usd:.2f} USD for order {order_id} "
        f"(reason: {reason}). Reference: REFUND-DEMO-0001."
    )


prompt = (
    "Please refund order ORDER-1234 for $29.99 — the customer was charged twice "
    "for their pro_plan subscription."
)

rprint("--- step 1: initial run ---")
first = await approval_agent.run(prompt)
if not isinstance(first.output, DeferredToolRequests):
    rprint("(no approvals needed — model answered directly)")
    rprint(first.output)
else:
    rprint(f"deferred approvals pending: {len(first.output.approvals)}")
    for call in first.output.approvals:
        rprint(f"  -> {call.tool_name}({call.args})")

    rprint("\n--- step 2: simulated approval ---")
    results = DeferredToolResults(
        approvals={call.tool_call_id: True for call in first.output.approvals},
    )
    rprint(f"approvals: {results.approvals}")

    rprint("\n--- step 3: resume ---")
    final = await approval_agent.run(
        message_history=first.all_messages(),
        deferred_tool_results=results,
    )
    output = final.output
    if isinstance(output, Answer):
        rprint(f"answer     : {output.text}")
        rprint(f"confidence : {output.confidence}")
        rprint(f"risk_flag  : {output.risk_flag}")
    else:
        rprint("(unexpected) still deferred:", output)

--- step 1: initial run ---

deferred approvals pending: 1

-> refund_order({'order_id': 'ORDER-1234', 'amount_usd': 29.99, 'reason': 'Customer was charged twice for 
pro_plan subscription'})

--- step 2: simulated approval ---

approvals: {'toolu_01G4sQaTuveLZxGFreowy2xR': True}

--- step 3: resume ---

answer     : The refund has been successfully processed for order ORDER-1234. A refund of $29.99 has been issued to
the customer for the duplicate charge on their pro_plan subscription. The refund reference number is 
REFUND-DEMO-0001.

confidence : 0.95

risk_flag  : none

## `agent.to_web()` — bundled chat UI

Returns a Starlette ASGI app that serves the official `@pydantic/ai-chat-ui` bundle and a streaming chat endpoint wired to your agent. We construct the app and print its type here — launching uvicorn inside the notebook would block the kernel.

NOTE: this works?

In [18]:
web_agent: Agent[str, Answer] = Agent(
    model,
    output_type=Answer,
    deps_type=str,
    name="web-assistant",
    instructions=SUPPORT_INSTRUCTIONS,
)

web_app = web_agent.to_web(deps="you")
rprint("app type :", type(web_app).__name__)
rprint("module   :", type(web_app).__module__)
rprint("\nlaunch with: uv run uvicorn your_module:web_app --reload")
rprint("then open  : http://localhost:8000")

app type : Starlette

module   : starlette.applications

launch with: uv run uvicorn your_module:web_app --reload

then open  : http://localhost:8000

## `agent.to_cli()` — Rich-powered REPL

Drops the agent into a streaming command-line chat. `to_cli()` returns a Typer app you can wire into your own CLI; `to_cli_sync()` runs it directly.

In [19]:
cli_agent: Agent[str, Answer] = Agent(
    model,
    output_type=Answer,
    deps_type=str,
    name="cli-assistant",
    instructions=SUPPORT_INSTRUCTIONS,
)

cli_app = cli_agent.to_cli(prog_name="support")
rprint("cli app type:", type(cli_app).__name__)
rprint("module      :", type(cli_app).__module__)
rprint('\nIn a script: cli_agent.to_cli_sync(prog_name="support") to drop into the chat.')

cli app type: coroutine

module      : builtins

In a script: cli_agent.to_cli_sync(prog_name="support") to drop into the chat.

## `agent.to_a2a()` — A2A protocol surface

Same agent, exposed as an [Agent2Agent](https://a2a-protocol.org/) endpoint: agent card at `/.well-known/agent.json`, JSON-RPC at the root. Requires the `fasta2a` extra to actually serve.

In [20]:
a2a_agent: Agent[None, Answer] = Agent(
    model,
    output_type=Answer,
    name="a2a-assistant",
    instructions=SUPPORT_INSTRUCTIONS,
)

a2a_app = a2a_agent.to_a2a(
    name="SupportAssistant",
    description="A SaaS support assistant exposed over the Agent2Agent protocol.",
    version="1.0.0",
    url="http://localhost:8001",
)
rprint("a2a app type:", type(a2a_app).__name__)
rprint("module      :", type(a2a_app).__module__)
rprint("\nlaunch with: uv run uvicorn your_module:a2a_app --port 8001")
rprint("then        : curl http://localhost:8001/.well-known/agent.json")

ImportError: Please install the `fasta2a` package to use `Agent.to_a2a()` method, you can use the `a2a` optional group — `pip install "pydantic-ai-slim[a2a]"`

## Durable execution — pointer only

For multi-hour or restart-survivable workflows, `pydantic-ai` integrates with three workflow runtimes. Same agent code, choose the engine:

- **[Temporal](https://temporal.io/)** — workflow-as-code, replayable, production-graded.
- **[DBOS](https://dbos.dev/)** — Postgres-backed, transactional, lightweight.
- **[Prefect](https://prefect.io/)** — orchestration platform with built-in observability.

Wire your agent via the durable-execution adapter and the runtime checkpoints state between steps — so a process restart picks up where the previous one left off. **Most chat-shaped agents don't need this** (request/response is short-lived). The primitive exists for the cases that do — multi-day approval workflows, periodic background research jobs, anything you can't lose.

## Recap

- **Test for free**: `TestModel`, `FunctionModel`, `capture_run_messages` — every `pytest` run, zero API spend.
- **Observe everything**: `configure_logfire()` + `instrument_pydantic_ai()` + `instrument_httpx()`. Free tier covers most workshop usage.
- **Eval on a schedule**: `Dataset` + custom `Evaluator`s. Pure / rule-based / LLM-judge cover most needs.
- **Production hardening**: `Hooks` for PII redaction and audit logging; `FallbackModel` for SLA insurance; `requires_approval=True` for human-in-the-loop.
- **Ship three ways**: `to_web()` (Starlette + chat UI), `to_cli()` (Typer + Rich), `to_a2a()` (FastA2A + JSON-RPC).
- **For long-running**: Temporal / DBOS / Prefect via the durable-execution adapter.

### The full feature tour

| Notebook | Theme |
|---|---|
| `01_intro` | Schemas, `Agent`, output modes, run methods. |
| `02_capabilities` | Tools, capabilities, builtin tools, MCP, RAG, multimodal. |
| `03_workflows` | Multi-agent, graphs, message history, compaction, memory. |
| `04_production` | Tests, observability, evals, hooks, fallback, deferred tools, deployment. |

You can now answer *"is there a `pydantic-ai` primitive for X?"* for most X you'll meet in practice.